# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score, precision_score, recall_score, roc_auc_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
import joblib
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from itertools import product
from tqdm import tqdm
import itertools
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJsZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [10]:
class FeatureExtractor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour
        df['weekday'] = df['timestamp'].dt.weekday
        df = df.drop(columns=['timestamp'])
        return df



class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, name_target: str):
        self.name_target = name_target
        self.ohe = OneHotEncoder(sparse=False, handle_unknown='ignore')
        self.cat_cols = []

    def fit(self, X, y=None):
        features = X.drop(columns=[self.name_target], errors='ignore')
        
        self.cat_cols = features.select_dtypes(include=['object', 'category']).columns.tolist()
        
        if self.cat_cols:
            self.ohe.fit(features[self.cat_cols])
        return self

    def transform(self, X):
        df = X.copy()
        
        target = df[self.name_target] if self.name_target in df.columns else None
        features = df.drop(columns=[self.name_target], errors='ignore')

        if self.cat_cols:
            encoded = self.ohe.transform(features[self.cat_cols])
            encoded_df = pd.DataFrame(
                encoded, 
                columns=self.ohe.get_feature_names(self.cat_cols),
                index=features.index
            )
            features = features.drop(columns=self.cat_cols)
            features = pd.concat([features, encoded_df], axis=1)

        if target is not None:
            return features, target
        return features


class TrainValidationTest:
    def __init__(self, test_size: float = 0.2, random_state: int = 21):
        self.test_size = test_size
        self.random_state = random_state

    def split(self, X, y):
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, 
            test_size=self.test_size, 
            random_state=self.random_state, 
            stratify=y
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_full, y_train_full, 
            test_size=self.test_size, 
            random_state=self.random_state, 
            stratify=y_train_full
        )
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [3]:
class ModelSelection():
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []

    def choose(self, X_train, y_train, X_valid, y_valid):
        self.results = []
        best_overall_score = -1.0
        best_overall_model_name = ""

        for idx, grid in enumerate(tqdm(self.grids, desc="Models Evaluation")):
            model_name = self.grid_dict[idx]
            print(f"Estimator: {model_name}")

            grid.fit(X_train, y_train)

            best_params = grid.best_params_
            best_train_acc = grid.best_score_
            valid_score = grid.score(X_valid, y_valid)

            self.results.append({
                'model': model_name,
                'params': best_params,
                'valid_score': valid_score
            })

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_train_acc:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}\n")

            if valid_score > best_overall_score:
                best_overall_score = valid_score
                best_overall_model_name = model_name

        print(f"Classifier with best validation set accuracy: {best_overall_model_name}")
        return best_overall_model_name

    
    def best_results(self):
        return pd.DataFrame(self.results)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [4]:
class Finalize():
    def __init__(self, estimator):
        self.estimator = estimator
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        
        y_pred = self.estimator.predict(X_test)
        score = accuracy_score(y_test, y_pred)
        
        print(f"Accuracy of the final model is {score}")
        return score

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print("Model successfully saved!")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [13]:
df = pd.read_csv('../data/checker_submits.csv')
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('weekday'))])
X, y = preprocessing.fit_transform(df)
X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest().split(X, y)

In [ ]:
param_grid_svc = {
    'kernel' : ['linear', 'rbf', 'sigmoid'],
    'C' : [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma' : ['scale', 'auto'],
    'class_weight':['balanced', None]
}
gs_svm = GridSearchCV(
    estimator=SVC(random_state=21),
    param_grid=param_grid_svc,
    scoring='accuracy',   
    n_jobs=-1,            
    verbose=1             
) # {'C': 10, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'rbf'}



param_grid_dt = {
    'max_depth' : range(1, 50),
    'class_weight':['balanced', None],
    'criterion':['entropy', 'gini']
}
gs_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=21),
    param_grid=param_grid_dt,
    scoring='accuracy',   
    n_jobs=-1,            
    verbose=1 
) # {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}



param_grid_forest = {
    'n_estimators' : [5, 10, 50, 100],
    'max_depth' : range(1, 50),
    'class_weight' : ['balanced', None],
    'criterion' : ['entropy', 'gini']
}
gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=21),
    param_grid=param_grid_forest,
    scoring='accuracy',   
    n_jobs=-1,            
    verbose=1 
)#  {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}

grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}
grids = [gs_svm, gs_tree, gs_rf]

# 2. Создание экземпляра и запуск
selector = ModelSelection(grids=grids, grid_dict=grid_dict)
best_model_name = selector.choose(X_train, y_train, X_valid, y_valid)

# 3. Получение сводного DataFrame
df_results = selector.best_results()
df_results

Models Evaluation:   0%|          | 0/3 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


Estimator: SVM
Fitting 5 folds for each of 72 candidates, totalling 360 fits


[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 284 tasks      | elapsed:   15.0s
[Parallel(n_jobs=-1)]: Done 360 out of 360 | elapsed:   36.1s finished
Models Evaluation:  33%|███▎      | 1/3 [00:36<01:12, 36.27s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:    0.0s


Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Best training accuracy: 0.842
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree
Fitting 5 folds for each of 196 candidates, totalling 980 fits


[Parallel(n_jobs=-1)]: Done 800 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 980 out of 980 | elapsed:    1.0s finished
Models Evaluation:  67%|██████▋   | 2/3 [00:37<00:25, 25.72s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 26}
Best training accuracy: 0.853
Validation set accuracy score for best params: 0.867

Estimator: Random Forest
Fitting 5 folds for each of 784 candidates, totalling 3920 fits


[Parallel(n_jobs=-1)]: Done  31 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 584 tasks      | elapsed:    5.9s
[Parallel(n_jobs=-1)]: Done 1584 tasks      | elapsed:   16.1s
[Parallel(n_jobs=-1)]: Done 2984 tasks      | elapsed:   31.1s
[Parallel(n_jobs=-1)]: Done 3920 out of 3920 | elapsed:   40.2s finished
Models Evaluation: 100%|██████████| 3/3 [01:17<00:00, 25.92s/it]

Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 35, 'n_estimators': 50}
Best training accuracy: 0.898
Validation set accuracy score for best params: 0.904

Classifier with best validation set accuracy: Random Forest


,model,params,valid_score
0,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.877778
1,Decision Tree,"{'class_weight': None, 'criterion': 'gini', 'm...",0.866667
2,Random Forest,"{'class_weight': 'balanced', 'criterion': 'ent...",0.903704


In [19]:
finalize = Finalize(RandomForestClassifier(class_weight = 'balanced', criterion = 'gini', max_depth = 38, n_estimators = 50, random_state=21))
final_score = finalize.final_score(X_train, y_train, X_test, y_test)
finalize.save_model(f"Random_forest_{final_score}.sav")

Accuracy of the final model is 0.9112426035502958
Model successfully saved!


In [20]:
rf_model = joblib.load('Random_forest_0.9112426035502958.sav')

In [22]:
final = Finalize(rf_model)
final.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.9112426035502958


0.9112426035502958